# A/B Testing — chiến dịch voucher

**Mục tiêu:** đo hiệu quả của voucher lên nhóm khách hàng mục tiêu (Cụm 1 — *Khách cuối tuần, trả Flex, tip thấp*)
| | |
|---|---|
| H₀ | Voucher không làm thay đổi số chuyến trung bình của rider thuộc cụm target |
| H₁ | Voucher làm tăng số chuyến trung bình |
| Primary metric | **Incremental trips per rider** — chênh lệch số chuyến/30 ngày giữa nhóm nhận và không nhận |
| Đơn vị ngẫu nhiên hoá | rider |

In [1]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

B    = r"c:/Users/Linh/Desktop/Growth & Experimentation Project for Ride-Hailing Promotions"
SRC  = os.path.join(B, "02. Synthetic_data", "outputs", "experiment_ab_final.csv")
SEG  = os.path.join(B, "03. Segmentation", "outputs", "rider_cluster.csv")
OUT  = os.path.join(B, "04. AB Testing", "outputs"); os.makedirs(OUT, exist_ok=True)

SURFACE, INK, SECOND, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SERIES, ALT     = "#e1e0d9", "#c3c2b7", "#2a78d6", "#eb6834"


df  = pd.read_csv(SRC)
seg = pd.read_csv(SEG)[["user_id", "cluster", "persona"]]

_before = len(df)
df = df.merge(seg, on="user_id", how="inner", validate="1:1")
assert len(df) == _before, f"merge mat dong: {_before:,} -> {len(df):,}"

In [2]:
TARGET_CLUSTER = 1
tg = df[df.cluster == TARGET_CLUSTER].copy()
TARGET = tg.persona.mode().iloc[0]          # ten mo ta, chi de hien thi
t_, c_ = tg[tg.T_rct == 1], tg[tg.T_rct == 0]

print(f"Cum {TARGET_CLUSTER} — {TARGET}")
print(f"   quy mo    : {len(tg):,} rider ({len(tg)/len(df):.1%} ")
print(f"   treatment : {len(t_):,}")
print(f"   control   : {len(c_):,}")

Cum 1 — Khach cuoi tuan, tra Flex, tip thap
   quy mo    : 6,078 rider (30.4% 
   treatment : 3,029
   control   : 3,049


## 1. Sanity check

In [3]:
from scipy.stats import binomtest

n_t, n_c = len(t_), len(c_)
srm = binomtest(n_t, n_t + n_c, 0.5)
print(f"Ty le chia nhom : {n_t:,} / {n_c:,}  ({n_t/(n_t+n_c):.1%} treatment)")
print(f"SRM binomial p  : {srm.pvalue:.4f}  ->  {'OK' if srm.pvalue > 0.001 else 'CO SRM - dung lai'}")

Ty le chia nhom : 3,029 / 3,049  (49.8% treatment)
SRM binomial p  : 0.8075  ->  OK


In [4]:
COV = ["total_rides", "recency_days", "typical_distance", "route_entropy",
       "pct_airport", "weekend_ratio", "pct_flex_payment", "pct_tip_rate",
       "avg_fare", "age", "is_urban"]

def smd(a, b):
    """Standardized Mean Difference — chenh lech chia do lech chuan gop."""
    s = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return 0.0 if s == 0 else (a.mean() - b.mean()) / s

bal = pd.DataFrame({
    "treatment": [t_[c].mean() for c in COV],
    "control":   [c_[c].mean() for c in COV],
    "SMD":       [smd(t_[c], c_[c]) for c in COV],
}, index=COV)
bal["|SMD|"] = bal.SMD.abs()
bal = bal.sort_values("|SMD|", ascending=False)
display(bal.round(4))
print(f"|SMD| lon nhat = {bal['|SMD|'].max():.4f}  ->  "
      f"{'CAN BANG' if bal['|SMD|'].max() < 0.1 else 'MAT CAN BANG'}")

,treatment,control,SMD,|SMD|
weekend_ratio,0.4144,0.4260,-0.0531,0.0531
route_entropy,1.8259,1.8087,0.0251,0.0251
total_rides,7.6124,7.5415,0.0152,0.0152
pct_flex_payment,0.4481,0.4503,-0.0112,0.0112
pct_tip_rate,0.2054,0.2044,0.0105,0.0105
typical_distance,2.7405,2.7586,-0.0103,0.0103
is_urban,0.8293,0.8262,0.0083,0.0083
recency_days,54.9333,55.3457,-0.0077,0.0077
age,41.1248,41.0521,0.0053,0.0053
avg_fare,22.1695,22.1361,0.0044,0.0044


|SMD| lon nhat = 0.0531  ->  CAN BANG
